# Nemotron LoRA Training — Fully Offline, RTX Pro 6000

**Required Kaggle inputs (add in sidebar before running):**
- `nemotron-offline-deps` dataset — contains wheelhouse (129 wheels) + NuminaMath parquet
- `huikang/nemotron-adapter` — competition LoRA adapter
- `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16` — base model (separate model input)

**Internet must be OFF.**

---

### All fixes applied vs original notebook

| # | Cell | Bug | Fix |
|---|------|-----|-----|
| 1 | 1 | Duplicate discover cell re-ran `find_wheelhouse` | Merged into single install cell |
| 2 | 1 | Stray wheel-list cell used `os` without importing | Folded into Cell 1 |
| 3 | 2 | Error message referenced wrong model name (`30B`) | Corrected to `4B` |
| 4 | 2 | `BASE_MODEL_PATH`/`ADAPTER_PATH` stayed `None` on unknown path type | Added `else: raise RuntimeError` |
| 5 | 3 | `PeftModel.from_pretrained` on raw 4-bit model | Use `load_adapter` directly |
| 6 | 6 | `eval_strategy=` rejected by transformers >=4.46 | Key is valid in unsloth SFTConfig |
| 7 | 6 | `SFTTrainer(tokenizer=...)` deprecated in trl >=0.9 | Changed to `processing_class=tokenizer` |
| 8 | 7 | `GRPOConfig(max_new_tokens=...)` not a valid arg | Changed to `max_completion_length=` |
| 9 | 7 | `GRPOConfig(temperature=...)` not a valid top-level arg | Moved to `generation_kwargs=` |
| 10 | 7 | `reward_correctness(solution=...)` crashes if column absent | Added `solution=None` default |
| 11 | 1 | numpy version mismatch (disk != memory) | Dist-info patch + sklearn removal |
| 12 | 1 | `unsloth` imported after `transformers` | Install unsloth first |
| 13 | 1 | `os._exit(0)` hard-kills kernel | Removed all restart logic |
| 14 | 3 | Double-wrapping LoRA + dtype mismatch in MoE `index_add_()` | `load_adapter` direct + bfloat16 cast |
| 15 | 3+6+7 | **Repeated `index_add_()` BFloat16/Float crash** — unsloth's SFTTrainer re-calls `for_training()` at `trainer.train()` start, resetting LoRA matrices to Float32 and undoing Fix 14 | Added `register_moe_hooks()` (forward hook on every MoE expert leaf, survives all `for_training()` re-entries) in Cell 3 **and** `BFloat16RecastCallback` (on_train_begin re-cast) in Cell 6 and Cell 7 |


## Cell 1 — Install all packages from local wheels (offline)


In [ ]:
import os, glob, json, subprocess, sys, re, shutil, site
import importlib.metadata as imd

def find_wheelhouse():
    for dirpath, _, files in os.walk('/kaggle/input'):
        if any(f.endswith('.whl') for f in files):
            return dirpath
    return None

WHEEL_DIR = find_wheelhouse()
assert WHEEL_DIR, 'Wheelhouse not found'
print(f'Wheelhouse: {WHEEL_DIR}')

with open(os.path.join(WHEEL_DIR, 'manifest.json')) as f:
    manifest = json.load(f)
print('manifest:', json.dumps(manifest, indent=2))

all_wheels = sorted(glob.glob(WHEEL_DIR + '/*.whl'))
print(f'Found {len(all_wheels)} wheels.')

for pkg_key in ['accelerate', 'peft']:
    matches = sorted(glob.glob(os.path.join(WHEEL_DIR, f'{pkg_key}*.whl')))
    print(f'  Wheelhouse {pkg_key}: {[os.path.basename(m) for m in matches]}')

import numpy as _np
MEM_NUMPY = _np.__version__
print(f'numpy in memory: {MEM_NUMPY}')


def install(spec, no_deps=False, force=False):
    cmd = [sys.executable, '-m', 'pip', 'install',
           '--no-index', f'--find-links={WHEEL_DIR}', spec, '-q']
    if no_deps: cmd.append('--no-deps')
    if force:   cmd.append('--force-reinstall')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        stderr = r.stderr.strip()
        if 'No matching distribution' in stderr and '==' in spec:
            pkg_name = spec.split('==')[0]
            print(f'  WARN: {spec} not in wheelhouse — retrying {pkg_name} (latest)')
            cmd2 = [sys.executable, '-m', 'pip', 'install',
                    '--no-index', f'--find-links={WHEEL_DIR}', pkg_name, '-q']
            if no_deps: cmd2.append('--no-deps')
            if force:   cmd2.append('--force-reinstall')
            r2 = subprocess.run(cmd2, capture_output=True, text=True)
            if r2.returncode == 0:
                try:    ver = imd.version(pkg_name)
                except: ver = '?'
                print(f'  OK: {pkg_name} (installed: {ver})')
                return True
            print(f'  ERROR {pkg_name}: {r2.stderr.strip()[:300]}')
            return False
        print(f'  ERROR {spec}: {stderr[:300]}')
        return False
    try:    ver = imd.version(spec.split('==')[0])
    except: ver = '?'
    print(f'  OK: {spec} (installed: {ver})')
    return True


for pkg in ['scikit-learn', 'scipy']:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', pkg, '-y', '-q'],
        capture_output=True, text=True)
    print(f'{pkg}: {"removed" if r.returncode == 0 else "already absent"}')

print('\nInstalling packages...')

unsloth_zoo_ver = manifest.get('unsloth_zoo', '')
install(f'unsloth_zoo=={unsloth_zoo_ver}' if unsloth_zoo_ver else 'unsloth_zoo')

unsloth_ver = manifest.get('unsloth', '')
install(f'unsloth=={unsloth_ver}' if unsloth_ver else 'unsloth', no_deps=True)

for pkg in ['huggingface_hub', 'bitsandbytes', 'accelerate', 'datasets', 'peft']:
    ver = manifest.get(pkg.lower(), '')
    install(f'{pkg}=={ver}' if ver else pkg)

install('trl', no_deps=True)

tf_ver = manifest.get('transformers', '5.5.0')
install(f'transformers=={tf_ver}', force=True)

for pattern, pip_name in [('causal_conv1d*.whl', 'causal-conv1d'),
                           ('mamba_ssm*.whl',     'mamba-ssm')]:
    matches = glob.glob(os.path.join(WHEEL_DIR, pattern))
    if matches:
        print(f'  Installing {pip_name}: {os.path.basename(matches[0])}')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install',
             matches[0], '--no-deps', '--no-index', '--force-reinstall', '-q'],
            capture_output=True, text=True)
        print(f'  {"OK" if r.returncode == 0 else "ERROR: " + r.stderr[:300]}')
    else:
        print(f'  WARNING: no wheel found for {pip_name}')


import importlib
importlib.invalidate_caches()

search_dirs = list(site.getsitepackages())
try:
    search_dirs.append(site.getusersitepackages())
except Exception:
    pass

numpy_distinfos = []
for d in search_dirs:
    numpy_distinfos.extend(glob.glob(os.path.join(d, 'numpy-*.dist-info')))
numpy_distinfos = sorted(set(numpy_distinfos))
print(f'\nnumpy dist-info dirs found: {[os.path.basename(d) for d in numpy_distinfos]}')

patched = False
for dist_dir in numpy_distinfos:
    disk_ver = os.path.basename(dist_dir).replace('numpy-', '').replace('.dist-info', '')
    if disk_ver == MEM_NUMPY:
        print(f'dist-info already matches in-memory version ({MEM_NUMPY}) — no patch needed.')
        patched = True
        break
    metadata_path = os.path.join(dist_dir, 'METADATA')
    if not os.path.exists(metadata_path):
        continue
    meta = open(metadata_path).read()
    meta_new = re.sub(r'^Version:.*$', f'Version: {MEM_NUMPY}', meta, flags=re.MULTILINE)
    open(metadata_path, 'w').write(meta_new)
    new_dir = os.path.join(os.path.dirname(dist_dir), f'numpy-{MEM_NUMPY}.dist-info')
    if os.path.exists(new_dir):
        shutil.rmtree(new_dir)
    os.rename(dist_dir, new_dir)
    print(f'Patched: {os.path.basename(dist_dir)} -> numpy-{MEM_NUMPY}.dist-info')
    patched = True
    break

if not patched:
    print(f'No numpy dist-info found anywhere — creating synthetic one for {MEM_NUMPY}')
    target_dir = os.path.join(site.getsitepackages()[0], f'numpy-{MEM_NUMPY}.dist-info')
    os.makedirs(target_dir, exist_ok=True)
    with open(os.path.join(target_dir, 'METADATA'), 'w') as fh:
        fh.write(f'Metadata-Version: 2.1\nName: numpy\nVersion: {MEM_NUMPY}\n')
    with open(os.path.join(target_dir, 'INSTALLER'), 'w') as fh:
        fh.write('pip\n')
    print(f'Created: {target_dir}')
    patched = True

importlib.invalidate_caches()
import importlib.metadata as imd2

disk_numpy_now = imd2.version('numpy')
print(f'numpy in memory  : {MEM_NUMPY}')
print(f'numpy on disk now: {disk_numpy_now}')
assert disk_numpy_now == MEM_NUMPY, f'Patch failed: disk shows {disk_numpy_now}'
print('numpy version match confirmed.')

print('\nFinal versions:')
for pkg in ['numpy', 'unsloth', 'unsloth_zoo', 'trl', 'peft',
            'transformers', 'accelerate', 'bitsandbytes',
            'mamba_ssm', 'causal_conv1d']:
    try:
        print(f'  {pkg}: {imd2.version(pkg)}')
    except Exception:
        print(f'  {pkg}: NOT FOUND')

import unsloth
import mamba_ssm, causal_conv1d
print(f'unsloth       : {unsloth.__version__}')
print(f'mamba_ssm     : {mamba_ssm.__version__}')
print(f'causal_conv1d : {causal_conv1d.__version__}')
print('\nCell 1 complete.')


## Cell 2 — Detect model type & locate base model

Inspects the competition path, finds the real base model, and patches `adapter_config.json` if `base_model_name_or_path` is null.


In [ ]:
import os, json, shutil

COMPETITION_PATH = '/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20'

files_here = os.listdir(COMPETITION_PATH) if os.path.isdir(COMPETITION_PATH) else []
print('Files at competition path:')
for f in sorted(files_here):
    print(f'  {f}')

IS_ADAPTER = 'adapter_config.json' in files_here
IS_BASE    = 'config.json' in files_here and not IS_ADAPTER
print(f'\nPath type: {"LoRA adapter" if IS_ADAPTER else "base model" if IS_BASE else "UNKNOWN"}')

BASE_MODEL_PATH = None
ADAPTER_PATH    = None

if IS_BASE:
    BASE_MODEL_PATH = COMPETITION_PATH
    print('Competition path is a base model — using directly.')

elif IS_ADAPTER:
    with open(os.path.join(COMPETITION_PATH, 'adapter_config.json')) as f:
        adapter_cfg = json.load(f)
    print('\nadapter_config.json:')
    print(json.dumps(adapter_cfg, indent=2))

    print('\nSearching for base model in /kaggle/input...')
    candidates = []
    for dirpath, _, dirfiles in os.walk('/kaggle/input'):
        if COMPETITION_PATH in dirpath:
            continue
        if 'config.json' in dirfiles and 'adapter_config.json' not in dirfiles:
            has_weights = any(
                f.endswith('.safetensors') or f.endswith('.bin') for f in dirfiles
            )
            if has_weights:
                try:
                    with open(os.path.join(dirpath, 'config.json')) as f:
                        cfg = json.load(f)
                    candidates.append((dirpath, cfg.get('model_type', '?')))
                except Exception:
                    pass

    print(f'Candidates found: {len(candidates)}')
    for p, mt in candidates:
        print(f'  [{mt}] {p}')

    if not candidates:
        raise RuntimeError(
            'No base model found in /kaggle/input!\n'
            'Add nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16 as a Kaggle model input.'
        )

    BASE_MODEL_PATH = candidates[0][0]
    print(f'\nSelected base model: {BASE_MODEL_PATH}')

    current_ref = adapter_cfg.get('base_model_name_or_path')
    if not current_ref or str(current_ref).lower() in ('none', 'null', ''):
        print(f'Patching adapter_config: base_model_name_or_path was {current_ref!r}')
        ADAPTER_COPY = '/kaggle/working/adapter_patched'
        if os.path.exists(ADAPTER_COPY):
            shutil.rmtree(ADAPTER_COPY)
        shutil.copytree(COMPETITION_PATH, ADAPTER_COPY)
        patched = dict(adapter_cfg)
        patched['base_model_name_or_path'] = BASE_MODEL_PATH
        with open(os.path.join(ADAPTER_COPY, 'adapter_config.json'), 'w') as f:
            json.dump(patched, f, indent=2)
        ADAPTER_PATH = ADAPTER_COPY
        print(f'Patched adapter written to: {ADAPTER_PATH}')
    else:
        print(f'base_model_name_or_path already set: {current_ref}')
        ADAPTER_PATH = COMPETITION_PATH

else:
    raise RuntimeError(
        f'Competition path is neither a base model nor a LoRA adapter.\n'
        f'Path: {COMPETITION_PATH}\n'
        f'Files found: {files_here}'
    )

print(f'\nBASE_MODEL_PATH = {BASE_MODEL_PATH}')
print(f'ADAPTER_PATH    = {ADAPTER_PATH}')


## Cell 3 — Load base model + competition adapter (4-bit QLoRA)

**Fix 14 + Fix 15:** `load_adapter` directly (no double-wrapping), bfloat16 cast, and MoE expert output hooks that survive `for_training()` re-entries.


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
LORA_RANK   = 32

# ── 1. Load base model ───────────────────────────────────────────────────────
print(f'Loading base model: {BASE_MODEL_PATH}')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    trust_remote_code=True,
)
print(f'Base loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

# ── 2. MoE expert dtype-safety hooks ────────────────────────────────────────
#
# FIX 15 (root cause of the repeated crash)
# ──────────────────────────────────────────
# Problem:
#   Even after Fix 14 (casting LoRA params to bfloat16 post-load), the crash
#   'RuntimeError: index_add_(): self (BFloat16) and source (Float) must have
#    the same scalar type' [modeling_nemotron_h.py:852] kept recurring.
#
# Root cause:
#   unsloth's SFTTrainer wrapper calls model.for_training() AGAIN internally
#   at the very start of trainer.train() (UnslothSFTTrainer.py line 83).
#   This re-enters unsloth's patching path, which can re-initialise LoRA A/B
#   matrices to Float32 — undoing the bfloat16 cast from Fix 14.
#   When the MoE expert forward then runs:
#     expert_output = expert(expert_input)        # Float32 (LoRA output)
#     weighted_output = expert_output * weights   # still Float32
#     final_hidden_states.index_add_(...)         # BFloat16 accumulator → CRASH
#
# Fix:
#   Register a persistent nn.Module forward hook on every individual expert
#   sub-module (leaf modules inside .experts containers). The hook casts the
#   expert's output tensor to bfloat16 before index_add_() sees it. Hooks
#   are attached to the nn.Module objects themselves and survive any number
#   of for_training() calls. Combined with BFloat16RecastCallback in Cell 6
#   (which re-casts param.data after trainer setup), this eliminates the crash.
#

def _cast_to_bf16_hook(module, args, output):
    if isinstance(output, torch.Tensor) and output.dtype != torch.bfloat16:
        return output.to(torch.bfloat16)
    return output

def register_moe_hooks(mdl):
    hooks = []
    for name, module in mdl.named_modules():
        parent = name.rsplit('.', 1)[0] if '.' in name else ''
        if 'experts' in parent and not list(module.children()):
            h = module.register_forward_hook(_cast_to_bf16_hook)
            hooks.append(h)
    return hooks

# ── 3. Wrap / load adapter ────────────────────────────────────────────────────
if ADAPTER_PATH:
    print(f'\nLoading competition adapter: {ADAPTER_PATH}')
    model.load_adapter(ADAPTER_PATH, adapter_name='default', is_trainable=True)

    cast_count = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype != torch.bfloat16:
            param.data = param.data.to(torch.bfloat16)
            cast_count += 1
    print(f'Cast {cast_count} trainable LoRA parameter tensors -> bfloat16.')

    model = FastLanguageModel.for_training(model)
    print('Competition adapter loaded — continuing training from checkpoint.')

else:
    print('No adapter path — applying fresh LoRA for scratch training.')
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=64,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype != torch.bfloat16:
            param.data = param.data.to(torch.bfloat16)
    print('Fresh LoRA applied.')

# Register MoE hooks — survive all subsequent for_training() calls
_moe_hooks = register_moe_hooks(model)
print(f'Registered {len(_moe_hooks)} MoE expert output hooks (bfloat16 guard).')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\nTrainable : {trainable / 1e6:.1f} M  ({100 * trainable / total:.2f}%)')
print(f'VRAM      : {torch.cuda.memory_allocated() / 1e9:.1f} GB')


## Cell 4 — Load NuminaMath dataset from local parquet (offline)


In [ ]:
from datasets import load_dataset
import os

def find_parquet(hint='numinamath'):
    for dirpath, _, files in os.walk('/kaggle/input'):
        if hint.lower() in dirpath.lower() or any(hint in f for f in files):
            for f in files:
                if f.endswith('.parquet'):
                    return os.path.join(dirpath, f)
    for dirpath, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.parquet'):
                return os.path.join(dirpath, f)
    return None

PARQUET_PATH = find_parquet()
assert PARQUET_PATH, (
    'NuminaMath parquet not found in /kaggle/input.\n'
    'Check that nemotron-offline-deps dataset is attached and contains a .parquet file.'
)
print(f'Loading: {PARQUET_PATH}')

ds = load_dataset('parquet', data_files={'train': PARQUET_PATH}, split='train')
print(f'Loaded  : {len(ds):,} examples')
print(f'Columns : {ds.column_names}')
print(f'\nSample problem:\n{ds[0]["problem"][:300]}')
print(f'\nSample solution (first 200 chars):\n{ds[0]["solution"][:200]}')


## Cell 5 — Format dataset into chat template


In [ ]:
SYSTEM_PROMPT = (
    'You are a mathematical reasoning expert. '
    'Solve problems step by step, showing all working. '
    'Always place your final answer inside \\boxed{} at the end.'
)

def format_for_sft(example):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': example['problem']},
        {'role': 'assistant', 'content': example['solution']},
    ]
    return {
        'text': tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

N_SFT = min(50_000, len(ds))
sft_raw  = ds.select(range(N_SFT))
grpo_raw = ds.select(range(N_SFT, len(ds)))

print(f'SFT pool  : {len(sft_raw):,} examples')
print(f'GRPO pool : {len(grpo_raw):,} examples')

ds_fmt = sft_raw.map(
    format_for_sft,
    num_proc=2,
    remove_columns=sft_raw.column_names,
    desc='Formatting',
)
print(f'\nFormatted : {len(ds_fmt):,} examples')
print(f'\nSample (first 500 chars):')
print(ds_fmt[0]['text'][:500])


## Cell 6 — SFT Training

Expected duration: ~4-6 hrs on RTX Pro 6000 (48 GB VRAM).

**Fix 15:** `BFloat16RecastCallback` added — re-casts LoRA params to bfloat16 after unsloth's internal `for_training()` call at training start.


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import torch

# ── FIX 15: BFloat16RecastCallback ────────────────────────────────────────────
# unsloth's SFTTrainer calls model.for_training() at trainer.train() start
# (UnslothSFTTrainer.py:83). This can re-initialise LoRA A/B matrices to
# Float32, undoing the bfloat16 cast from Cell 3.
# on_train_begin runs AFTER for_training(), so this re-cast always wins.

class BFloat16RecastCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        cast_count = 0
        for name, param in model.named_parameters():
            if param.requires_grad and param.dtype != torch.bfloat16:
                param.data = param.data.to(torch.bfloat16)
                cast_count += 1
        if cast_count:
            print(f'[BFloat16RecastCallback] Re-cast {cast_count} LoRA params '
                  f'-> bfloat16 (after for_training).')


split    = ds_fmt.train_test_split(test_size=0.01, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print(f'Train : {len(train_ds):,}  |  Eval : {len(eval_ds):,}')

sft_args = SFTConfig(
    output_dir='/kaggle/working/checkpoints',

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,

    num_train_epochs=2,
    warmup_ratio=0.05,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',

    optim='adamw_8bit',
    weight_decay=0.01,
    max_grad_norm=1.0,

    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=True,

    bf16=True,
    tf32=True,

    logging_steps=25,
    report_to='none',
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_args,
    callbacks=[BFloat16RecastCallback()],
)

print(f'VRAM before training : {torch.cuda.memory_allocated() / 1e9:.1f} GB')
print('Starting SFT...')
trainer.train()
print('\nSFT complete!')


## Cell 7 — GRPO Reward Fine-tuning *(optional — run after SFT)*

Two reward signals: correctness (1.0) and format (0.2).

**Fix 15 applied here too:** `BFloat16RecastCallback` in `GRPOTrainer`.


In [ ]:
import re
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_training(model)

# Re-cast immediately after for_training() in case it reset any LoRA params
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.bfloat16:
        param.data = param.data.to(torch.bfloat16)


# ── FIX 15 (GRPO): BFloat16RecastCallback ─────────────────────────────────────
class BFloat16RecastCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        cast_count = 0
        for name, param in model.named_parameters():
            if param.requires_grad and param.dtype != torch.bfloat16:
                param.data = param.data.to(torch.bfloat16)
                cast_count += 1
        if cast_count:
            print(f'[BFloat16RecastCallback] Re-cast {cast_count} params -> bfloat16.')


# ── reward helpers ─────────────────────────────────────────────────────────────
def extract_boxed(text):
    m = re.search(r'\\boxed\{', text)
    if not m:
        return ''
    start, depth = m.end(), 1
    for i, ch in enumerate(text[start:]):
        depth += (ch == '{') - (ch == '}')
        if depth == 0:
            return text[start:start + i].strip()
    return ''

def num_eq(a, b, tol=1e-6):
    try:
        fa = float(a.replace(',', ''))
        fb = float(b.replace(',', ''))
        return abs(fa - fb) / max(abs(fb), 1e-9) < tol
    except ValueError:
        return False

def reward_correctness(completions, solution=None, **kwargs):
    if solution is None:
        return [0.0] * len(completions)
    gold = extract_boxed(solution[0] if isinstance(solution, list) else solution)
    rewards = []
    for c in completions:
        pred = extract_boxed(c)
        if not pred:
            rewards.append(0.0)
        elif pred == gold or num_eq(pred, gold):
            rewards.append(1.0)
        else:
            rewards.append(0.1)
    return rewards

def reward_format(completions, **kwargs):
    return [0.2 if re.search(r'\\boxed\{', c) else 0.0 for c in completions]


# ── format GRPO dataset ────────────────────────────────────────────────────────
def fmt_grpo(x):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': x['problem']},
    ]
    return {
        'prompt':   tokenizer.apply_chat_template(
                        msgs, tokenize=False, add_generation_prompt=True),
        'solution': x['solution'],
    }

grpo_ds = grpo_raw.map(fmt_grpo, remove_columns=grpo_raw.column_names, desc='GRPO format')
print(f'GRPO dataset: {len(grpo_ds):,} examples')


# ── GRPO config ────────────────────────────────────────────────────────────────
grpo_cfg = GRPOConfig(
    output_dir='/kaggle/working/grpo_checkpoints',

    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=0.3,
    warmup_ratio=0.1,

    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to='none',

    num_generations=4,
    max_completion_length=1024,
    generation_kwargs={'temperature': 0.7, 'do_sample': True},
    seed=42,
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_correctness, reward_format],
    args=grpo_cfg,
    train_dataset=grpo_ds,
    callbacks=[BFloat16RecastCallback()],
)

print('Starting GRPO...')
grpo_trainer.train()
print('\nGRPO complete!')


## Cell 8 — Save adapter & verify competition constraints


In [ ]:
import os, shutil, json

ADAPTER_OUT = '/kaggle/working/nemotron-adapter-ready-to-submit'
if os.path.exists(ADAPTER_OUT):
    shutil.rmtree(ADAPTER_OUT)
os.makedirs(ADAPTER_OUT)

model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f'Saved to: {ADAPTER_OUT}')

cfg_path = os.path.join(ADAPTER_OUT, 'adapter_config.json')
with open(cfg_path) as f:
    cfg = json.load(f)
print('\nadapter_config.json:')
print(json.dumps(cfg, indent=2))

rank = cfg.get('r', 999)
assert rank <= 32, (
    f'Rank {rank} exceeds competition limit of 32!\n'
    f'Retrain with LORA_RANK <= 32.'
)
print(f'\n OK Rank check passed: r={rank} <= 32')

print('\nFiles:')
total_mb = 0
for fn in sorted(os.listdir(ADAPTER_OUT)):
    sz = os.path.getsize(os.path.join(ADAPTER_OUT, fn)) / 1e6
    total_mb += sz
    print(f'  {fn:45s} {sz:6.1f} MB')
print(f'  {"TOTAL":45s} {total_mb:6.1f} MB')


## Cell 9 — Zip and submit


In [ ]:
import shutil, os

ZIP_BASE = '/kaggle/working/submission'
zip_path = shutil.make_archive(ZIP_BASE, 'zip', ADAPTER_OUT)
size_mb  = os.path.getsize(zip_path) / 1e6

print(f'Created : {zip_path}')
print(f'Size    : {size_mb:.1f} MB')
print('\nSubmit /kaggle/working/submission.zip to the competition.')
print('Done')
